# Building an FRTB Market Risk Engine

A system that measures market risk with Value at Risk (VaR) and Expected Shortfall (ES), validates the model with the full regulatory backtesting suite, calibrates a stressed-period capital number, and runs a P&L Attribution Test — the four pillars of the Basel FRTB Internal Models Approach.

### Background and theory

#### Purpose: 
Banks must hold capital against the risk that their trading positions lose money. To size that capital they need a number that summarises "how bad can a bad day be." For three decades that number was VaR. After VaR failed repeatedly in crises (LTCM 1998, the 2008 crisis, the 2022 rate shock), the Basel Committee's *Fundamental Review of the Trading Book* (FRTB) replaced it with Expected Shortfall and wrapped it in a stricter validation regime. This project builds a miniature, honest version of that regime so we understand each piece from the inside.

#### Value at Risk

In shorts, it quantifies the potential loss for given confidence level and time horizon. 

Look at the left-side tail, In 1-day 99% VaR of $1 million, there is a 1% chance the daily loss will exceed $1 million.

Limitations: VaR does not capture the severity of losses beyond the threshold. It is not a coherent risk measure. Only tell where the tail begins but says nothing about how heavy the tail is beyond that point. 

#### Expected Shortfall

ES (CVaR) averages losses beyond the VaR threshold for given confidence level.

FRTB uses $\alpha = 97.5\%$ for ES. By construction $\text{ES}_\alpha \ge \text{VaR}_\alpha$ always, and the ratio $\text{ES}/\text{VaR}$ is a direct measure of tail heaviness. ES is also *coherent* (it satisfies sub-additivity: diversification can never increase it), a mathematical property VaR lacks.

### The Four Pillars will be built in the project

1. **Measurement** - compute VaR and ES from data
2. **Backtesting** - prove statistically that the model is accurate
3. **Stressed calibration** - size capital against the worst historical window, not a calm one. 
4. **P&L Attribution** - prove the model actually describes the portfolio it claims to. 


I will use a three-asset portfolio with genuinely different risk behaviours: an equity ETF (SPY), a long-Treasury ETF (TLT), and gold (GLD), weighted 50/30/20.

In [ ]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy import stats
from datetime import datetime, date

 
TICKERS = ["SPY", "TLT", "GLD"]
WEIGHTS = np.array([0.5, 0.3, 0.2])
 
 
def _download_real(start, end):
    import yfinance as yf
    prices = yf.download(TICKERS, start=start, end=end, progress=False)["Close"]
    prices = prices[TICKERS].dropna()
    if len(prices) < 500:
        raise RuntimeError("Not enough data returned.")
    return prices.pct_change().dropna()
 
 
def _synthetic(n_days=2500, seed=42):
    """Correlated, fat-tailed returns with two embedded crisis windows."""
    rng = np.random.default_rng(seed)
    daily_vol = np.array([0.18, 0.12, 0.15]) / np.sqrt(252)
    daily_mean = np.array([0.07, 0.02, 0.04]) / 252
    corr = np.array([[1.00, -0.30, 0.10],
                     [-0.30, 1.00, 0.05],
                     [0.10, 0.05, 1.00]])
    chol = np.linalg.cholesky(corr)
 
    df = 4
    z = rng.standard_t(df=df, size=(n_days, 3)) * np.sqrt((df - 2) / df)
    z = z @ chol.T
 
    vol_mult = np.ones(n_days)
    vol_mult[800:860] = 2.8      # crisis 1
    vol_mult[1700:1775] = 3.5    # crisis 2 (the worst)
 
    returns = daily_mean + (z * daily_vol) * vol_mult[:, None]
    dates = pd.bdate_range("2015-01-02", periods=n_days)
    return pd.DataFrame(returns, index=dates, columns=TICKERS)
 
import datetime as dt 

def get_portfolio_returns(start="2015-01-01", end=dt.date.today().isoformat(), verbose=True):
    try:
        asset_returns = _download_real(start, end)
        source = "real market data (yfinance)"
    except Exception as e:
        asset_returns = _synthetic()
        source = f"synthetic fallback ({type(e).__name__})"
 
    port = asset_returns.values @ WEIGHTS
    port = pd.Series(port, index=asset_returns.index, name="port_return")
    if verbose:
        print(f"[data] source: {source}")
        print(f"[data] {len(port)} days, {port.index[0].date()} to {port.index[-1].date()}")
    return port, asset_returns
 
 
if __name__ == "__main__":
    r, _ = get_portfolio_returns()
    print(r.describe())


note: The synthetic generator uses a Cholesky decomposition of the correlation matrix to produce correlated draws, Student-$t$ innovations (degrees of freedom $= 4$) rescaled to unit variance via the factor $\sqrt{(df-2)/df}$ for genuine fat tails, and two embedded high-volatility windows so the stressed-window finder later has something meaningful to discover. The equity/bond correlation is set negative ($-0.30$) to mirror the historical flight-to-quality relationship.

### **1. Measurement: VaR and ES engine**
#### Estimation method

Use **historical simulation**: the empirical distribution of past losses is the forecast for tomorrow. No distributional assumption is made — the data speaks for itself. The historical VaR is simply the empirical $\alpha$-quantile of the loss window, and the historical ES is the mean of all losses at or above that quantile.

In [ ]:
def historical_var(losses, alpha=0.99):
    """Historical-simulation VaR."""
    return np.quantile(losses, alpha)
 
 
def historical_es(losses, alpha=0.975):
    """Historical-simulation ES."""
    losses = np.asarray(losses,dtype=float)
    var = historical_var(losses, alpha)
    tail_losses = losses[losses >= var]
    return tail_losses.mean() if tail_losses.size else var
 
 
def rolling_risk(returns, window=250, var_alpha=0.99, es_alpha=0.975):
    """One-day-ahead rolling VaR/ES with realized next-day loss."""
    losses = -returns.values
    var_series, es_series, realized = [], [], []
    for t in range(window, len(losses)):
        win = losses[t - window:t]
        var_series.append(historical_var(win, var_alpha))
        es_series.append(historical_es(win, es_alpha))
        realized.append(losses[t])
    return (np.array(var_series), np.array(es_series),
            np.array(realized), returns.index[window:])
 
 
def rolling_risk(returns, window=250, levels=(0.975, 0.99)):
    """
    One‑day‑ahead rolling VaR and ES at given confidence levels.
    Vectorised using sliding_window_view.

    Returns a dict with keys:
        'var' : {level: np.ndarray}
        'es'  : {level: np.ndarray}
        'realized' : np.ndarray of next‑day losses
        'dates'    : DatetimeIndex aligned to forecasts
    """
    losses = -np.asarray(returns.values, dtype=float)
    if len(losses) <= window:
        raise ValueError(f"Need more than {window} observations; got {len(losses)}.")

    # windows[i] is the trailing block used to forecast day (window + i)
    windows = sliding_window_view(losses, window)[:-1]   # drop last (no next-day loss)
    realized = losses[window:]
    dates = returns.index[window:]

    var, es = {}, {}
    for a in levels:
        var[a] = np.quantile(windows, a, axis=1)
        # ES = mean of losses in each window that are >= that window's VaR
        mask = windows >= var[a][:, None]
        masked = np.where(mask, windows, np.nan)
        es[a] = np.nanmean(masked, axis=1)

    return {"var": var, "es": es, "realized": realized, "dates": dates}

Interpretation: 
- This is a moderately risky, normal-looking diversified portfolio. 
- With 99% VaR of 1.73%, on roughly 1/100 day, the portfolio loses at least $1,730 on $100,000 worth portfolio.
- With 99% ES of 2.09%, on the bad days, the average loss is about 2.09% ($2,090 on $100,000) which is more painful than the VaR predicted. 
- Tail-heaviness ratio: 1.21x. When losses break through warning line, they are on average about %21 bigger than the line. 
- FRTB regime ratio: At 0.98, they give most identical answer for the portfolio, so switching methods wouldn't change much. 
Exceptions 34/2615 (expected 26.2): reality broke the warning line a bit more often tan predicted. The actual risk is higher than the expected risk: happended 34 times over 2615 days. 

Like all real markets, it occasionally delivers days noticeably worse than a simple model expects, which is precisely the blind spot this whole project is built to expose and measure.

### **2. Backtesting**

##### The Basel Traffic Light Test
 
Count exceptions (days where realised loss exceeded VaR) over a year and assign a zone. With a correct 99% model you expect 2.5 exceptions per 250 days. Up to 4 is sampling noise (green); 5–9 earns a capital add-on (yellow); 10+ rejects the model (red).



In [ ]:
def traffic_light(exceptions_count, days=250):
    scaled = round(exceptions_count * 250 / days)
    if scaled <= 4:
        return "GREEN", scaled
    elif scaled <= 9:
        return "YELLOW", scaled
    return "RED", scaled
    
def rolling_traffic_light(exceptions):
    """Traffic light using only the most recent 250 days."""
    last = exceptions[-250:] if len(exceptions) >= 250 else exceptions
    return traffic_light(int(last.sum()), len(last))

##### Kupiec's Proportion-of-Failures Test (unconditional coverage)
 
A proper likelihood-ratio test of whether the observed exception rate matches the claimed rate $p = 1-\alpha$. With $x$ exceptions in $n$ days and $\hat\pi = x/n$:
 
$$LR_{POF} = -2\ln\!\left[\frac{(1-p)^{\,n-x}\,p^{\,x}}{(1-\hat\pi)^{\,n-x}\,\hat\pi^{\,x}}\right] \;\sim\; \chi^2_1$$
 
A p-value below 0.05 rejects the null that the model is correctly calibrated.

In [ ]:
from scipy import stats
 
 
def kupiec_pof(exceptions_count, total_days, p=0.01):
    """Kupiec Proportion‑of‑Failures test (unconditional coverage)."""
    x, n = int(exceptions_count), int(total_days)
    if n == 0:
        return np.nan, np.nan
    pi_hat = x / n
    if x == 0:
        lr = -2.0 * n * np.log(1 - p)
    elif x == n:
        lr = -2.0 * (n * np.log(p) - n * np.log(pi_hat))
    else:
        log_l_null = (n - x) * np.log(1 - p) + x * np.log(p)
        log_l_alt = (n - x) * np.log(1 - pi_hat) + x * np.log(pi_hat)
        lr = -2.0 * (log_l_null - log_l_alt)
    return lr, 1 - stats.chi2.cdf(lr, df=1)

##### Christoffersen's Independence Test
 
Correct exception *frequency* is not enough; exceptions must also be *independent in time*. A good model spreads breaches evenly; a bad one lets them cluster during volatile stretches. Define the transition counts $n_{ij}$ (from state $i$ today to state $j$ tomorrow, where 1 = exception), with $\pi_{01}=n_{01}/(n_{00}+n_{01})$, $\pi_{11}=n_{11}/(n_{10}+n_{11})$, and overall $\pi$:
 
$$LR_{ind} = -2\ln\!\left[\frac{(1-\pi)^{\,n_{00}+n_{10}}\,\pi^{\,n_{01}+n_{11}}}{(1-\pi_{01})^{\,n_{00}}\,\pi_{01}^{\,n_{01}}\,(1-\pi_{11})^{\,n_{10}}\,\pi_{11}^{\,n_{11}}}\right] \;\sim\; \chi^2_1$$


In [ ]:
def christoffersen_independence(exceptions):
    """Christoffersen independence test (clustering of exceptions)."""
    e = np.asarray(exceptions).astype(int)
    n00 = n01 = n10 = n11 = 0
    for i in range(1, len(e)):
        prev, curr = e[i-1], e[i]
        if prev == 0 and curr == 0:
            n00 += 1
        elif prev == 0 and curr == 1:
            n01 += 1
        elif prev == 1 and curr == 0:
            n10 += 1
        else:
            n11 += 1

    pi01 = n01 / (n00 + n01) if (n00 + n01) else 0.0
    pi11 = n11 / (n10 + n11) if (n10 + n11) else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11) if (n00 + n01 + n10 + n11) else 0.0

    safe = lambda v: v if v > 0 else 1e-10
    log_l_null = (n00 + n10) * np.log(safe(1 - pi)) + (n01 + n11) * np.log(safe(pi))
    log_l_alt = (n00 * np.log(safe(1 - pi01)) + n01 * np.log(safe(pi01))
                 + n10 * np.log(safe(1 - pi11)) + n11 * np.log(safe(pi11)))
    lr = -2.0 * (log_l_null - log_l_alt)
    return lr, 1 - stats.chi2.cdf(lr, df=1)


The intuition: if $\pi_{11}$ (chance of an exception right after an exception) far exceeds $\pi_{01}$, breaches are clustering — the model is not reacting to rising volatility. That is the LTCM/2008 failure mode.

##### Conditional Coverage (the combined test)
 
**Christoffersen's full test** adds the two statistics; under the null it is $\chi^2_2$.
 
$$LR_{cc} = LR_{POF} + LR_{ind} \;\sim\; \chi^2_2$$

In [ ]:
def conditional_coverage(exceptions, total_days, p=0.01):
    """Christoffersen conditional coverage = frequency + independence."""
    x = int(np.asarray(exceptions).sum())
    lr_pof, _ = kupiec_pof(x, total_days, p)
    lr_ind, _ = christoffersen_independence(exceptions)
    lr_cc = lr_pof + lr_ind
    return lr_cc, 1 - stats.chi2.cdf(lr_cc, df=2)


##### The Acerbi-Szekely Test for Expected Shortfall
 
VaR is easy to backtest (count breaches); ES is not, because it is an average of a rarely-observed tail. The Acerbi-Szekely (2014) $Z_2$ statistic made ES backtesting practical. With indicator $I_t = \mathbf{1}\{L_t > \text{VaR}_t\}$:
 
$$Z_2 = \frac{1}{n\,(1-\alpha)}\sum_{t=1}^{n}\frac{L_t\,I_t}{\text{ES}_t} \;-\; 1$$
 
$Z_2 \approx 0$ means ES is accurate; $Z_2 \gg 0$ means realised tail losses exceed predicted ES (the model underestimates risk — the dangerous direction).

In [ ]:
def acerbi_szekely_Z2(realized, var, es, alpha):
    """Z2 statistic. VaR and ES must be at the same alpha."""
    realized, var, es = map(np.asarray, (realized, var, es))
    n = len(realized)
    ind = (realized > var).astype(float)
    return float((realized * ind / (es * (1 - alpha) * n)).sum() - 1)


def acerbi_szekely_Z1(realized, var, es, alpha):
    """Z1 statistic (conditional on a breach having occurred)."""
    realized, var, es = map(np.asarray, (realized, var, es))
    ind = realized > var
    nb = int(ind.sum())
    if nb == 0:
        return 0.0
    return float((realized[ind] / es[ind]).sum() / nb - 1)


def acerbi_szekely_pvalues(returns, window=250, alpha=0.975, n_sims=2000, seed=0):
    """
    Monte‑Carlo p‑values for Z1 and Z2 under the null that the model is correct.
    Small p‑value => ES underestimates tail risk.
    """
    rng = np.random.default_rng(seed)
    losses = -np.asarray(returns.values, float)

    r = rolling_risk(returns, window=window, levels=(alpha,))
    var = r["var"][alpha]
    es = r["es"][alpha]
    realized = r["realized"]
    n = len(var)

    windows = sliding_window_view(losses, window)[:-1]   # same windows as rolling_risk
    z1_obs = acerbi_szekely_Z1(realized, var, es, alpha)
    z2_obs = acerbi_szekely_Z2(realized, var, es, alpha)

    z1_sims = np.empty(n_sims)
    z2_sims = np.empty(n_sims)
    for s in range(n_sims):
        idx = rng.integers(0, window, size=n)
        sim_real = windows[np.arange(n), idx]
        z1_sims[s] = acerbi_szekely_Z1(sim_real, var, es, alpha)
        z2_sims[s] = acerbi_szekely_Z2(sim_real, var, es, alpha)

    return {
        "Z1": z1_obs, "Z1_p": float((z1_sims >= z1_obs).mean()),
        "Z2": z2_obs, "Z2_p": float((z2_sims >= z2_obs).mean()),
    }



##### Berkowitz density‑forecast test 

In [ ]:
def berkowitz_tail(returns, window=250, alpha=0.99, n_sims=2000, seed=1):
    """
    Berkowitz tail test: maps realised losses to normal quantiles under the
    empirical predictive distribution and tests for zero mean (LR test).
    """
    losses = -np.asarray(returns.values, float)
    r = rolling_risk(returns, window=window, levels=(alpha,))
    realized = r["realized"]
    n = len(realized)
    windows = sliding_window_view(losses, window)[:-1]

    # Probability integral transform (with continuity correction)
    u = (windows <= realized[:, None]).mean(axis=1)
    u = np.clip(u, 1.0 / (window + 1), window / (window + 1))
    z = stats.norm.ppf(u)

    mu = z.mean()
    var_z = z.var(ddof=1)
    lr = n * mu**2 / var_z if var_z > 0 else 0.0
    p_chi2 = 1 - stats.chi2.cdf(lr, df=1)

    # Monte‑Carlo p‑value for mean = 0
    rng = np.random.default_rng(seed)
    mus = np.empty(n_sims)
    for s in range(n_sims):
        idx = rng.integers(0, window, size=n)
        sim_real = windows[np.arange(n), idx]
        usim = (windows <= sim_real[:, None]).mean(axis=1)
        usim = np.clip(usim, 1.0 / (window + 1), window / (window + 1))
        mus[s] = stats.norm.ppf(usim).mean()
    p_mc = float((np.abs(mus) >= abs(mu)).mean())

    return {"mean_z": float(mu), "LR": float(lr), "p_chi2": float(p_chi2), "p_mc": p_mc}

##### Bootstrap confidence interval for a risk statistic

In [ ]:
def bootstrap_ci(losses, func, alpha=0.975, n_boot=2000, ci=0.95, seed=2):
    """Percentile bootstrap CI for a risk statistic (e.g. ES)."""
    rng = np.random.default_rng(seed)
    losses = np.asarray(losses, float)
    n = len(losses)
    est = func(losses, alpha)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        sample = losses[rng.integers(0, n, size=n)]
        boots[b] = func(sample, alpha)
    lo = np.quantile(boots, (1 - ci) / 2)
    hi = np.quantile(boots, 1 - (1 - ci) / 2)
    return {"estimate": float(est), "lo": float(lo), "hi": float(hi)}

##### Orchestrator

In [ ]:
def run_all(returns, window=250, p=0.01):
    """Run full backtesting suite. ES tests use same alpha (97.5%)."""
    r = rolling_risk(returns, window=window, levels=(0.975, 0.99))
    var99 = r["var"][0.99]
    var975 = r["var"][0.975]
    es975 = r["es"][0.975]
    realized, dates = r["realized"], r["dates"]

    exc99 = realized > var99
    x, n = int(exc99.sum()), len(realized)

    zone, scaled = traffic_light(x, n)
    roll_zone, roll_scaled = rolling_traffic_light(exc99)
    _, p_pof = kupiec_pof(x, n, p)
    _, p_ind = christoffersen_independence(exc99)
    _, p_cc = conditional_coverage(exc99, n, p)

    # Same‑alpha ES backtest at 97.5%
    z1_same = acerbi_szekely_Z1(realized, var975, es975, 0.975)
    z2_same = acerbi_szekely_Z2(realized, var975, es975, 0.975)

    return {
        "n": n, "exceptions": x, "expected": p * n,
        "traffic_light": zone, "scaled_exceptions": scaled,
        "rolling_traffic_light": roll_zone, "rolling_scaled": roll_scaled,
        "kupiec_p": p_pof, "independence_p": p_ind, "cond_cov_p": p_cc,
        "as_Z1": z1_same, "as_Z2": z2_same,
        "var99": var99, "var975": var975, "es975": es975,
        "realized": realized, "dates": dates, "exceptions_arr": exc99,
    }

### **3.Stressed-window calibration**

The regulatory purpose
 
Under the old rules a bank could calibrate on a recent calm period and report low risk. FRTB closes this by requiring the model to find and use its **worst** 250-day window for the *current* portfolio. You slide a window across all history and keep the one that maximises ES.

In [ ]:
CRISES = [
    ("2015-08-01", "2016-02-29", "China devaluation / oil crash"),
    ("2018-01-15", "2018-02-28", "Volmageddon (VIX spike)"),
    ("2018-10-01", "2018-12-31", "Q4-2018 selloff"),
    ("2020-02-15", "2020-04-30", "COVID-19 crash"),
    ("2022-01-01", "2022-10-31", "2022 rate-hike bond rout"),
    ("2023-03-01", "2023-03-31", "SVB / regional-bank stress"),
]


def _label_window(start, end):
    """Return crisis description whose period overlaps the window most."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    best_overlap, best_desc = pd.Timedelta(0), None
    for cs, ce, desc in CRISES:
        cs, ce = pd.Timestamp(cs), pd.Timestamp(ce)
        overlap = min(e, ce) - max(s, cs)
        if overlap > best_overlap:
            best_overlap, best_desc = overlap, desc
    return best_desc


def find_stressed_window(returns, window=250, alpha=0.975):
    """Find the 250‑day window with the highest ES at alpha."""
    losses = -np.asarray(returns.values, float)
    if len(losses) <= window:
        raise ValueError("Not enough data for stress window.")

    windows = sliding_window_view(losses, window)
    var = np.quantile(windows, alpha, axis=1)
    masked = np.where(windows >= var[:, None], windows, np.nan)
    es = np.nanmean(masked, axis=1)

    best = int(np.argmax(es))
    start, end = returns.index[best], returns.index[best + window - 1]
    current_es = historical_es(losses[-window:], alpha)

    return {
        "stressed_es": float(es[best]),
        "start": start, "end": end,
        "label": _label_window(start, end),
        "current_es": float(current_es),
        "stress_multiplier": float(es[best] / current_es),
    }


**The stress multiplier**

$$\text{Stress multiplier} = \frac{\text{Stressed ES}}{\text{Current ES}}$$
 
This is the headline number for explaining FRTB's intent. A multiplier of 2–3x means a calm-period calibration would have understated required capital by that factor. On real data the window will typically land on the March 2020 COVID crash.
 

### **4.The P&L Attribution Test (PLAT)**

PLAT decides whether a bank may use its internal model at all. The model produces a *theoretical* daily P&L from its risk factors; the desk produces an *actual* P&L. If the two series do not track closely, the model is not describing the portfolio, and the bank is forced onto the punitive Standardised Approach.

We use a simplified two-metric version: the Spearman rank correlation between the two P&L series, and the variance ratio of unexplained P&L,
 
$$\text{variance ratio} = \frac{\text{Var}(\text{actual} - \text{risk})}{\text{Var}(\text{actual})}$$

In [ ]:
def sensitivity_pnl(asset_returns, weights=WEIGHTS):
    """Theoretical P&L from linear (delta) sensitivities."""
    return np.asarray(asset_returns.values, float) @ weights


def plat_test(risk_pnl, actual_pnl):
    """
    Simplified FRTB PLAT using Spearman correlation and unexplained variance ratio.
    """
    risk_pnl = np.asarray(risk_pnl, float)
    actual_pnl = np.asarray(actual_pnl, float)
    corr, _ = stats.spearmanr(risk_pnl, actual_pnl)
    var_ratio = np.var(actual_pnl - risk_pnl) / np.var(actual_pnl)
    if corr > 0.80 and var_ratio < 0.10:
        zone = "GREEN  - internal model approved"
    elif corr > 0.70 and var_ratio < 0.20:
        zone = "AMBER  - capital surcharge"
    else:
        zone = "RED    - forced onto Standardised Approach"
    return {"spearman": float(corr), "variance_ratio": float(var_ratio), "zone": zone}


def plat_from_portfolio(returns, asset_returns, unexplained_sd_frac=0.08, seed=0):
    """Build realistic PLAT: theoretical P&L vs actual (true + small residual)."""
    rng = np.random.default_rng(seed)
    theo = sensitivity_pnl(asset_returns)
    true_pnl = np.asarray(returns.values, float)
    resid = rng.normal(0, true_pnl.std() * unexplained_sd_frac, len(true_pnl))
    actual = true_pnl + resid
    return plat_test(theo, actual)

### The daily risk report


In [ ]:

def generate_report(with_pvalues=False):
    """Print a formatted daily risk report."""
    returns, asset_returns = get_portfolio_returns(verbose=False)
    bt = run_all(returns)
    sw = find_stressed_window(returns)
    plat = plat_from_portfolio(returns, asset_returns)

    # Bootstrap CI for the latest 250‑day window ES
    last_win = -np.asarray(returns.values, float)[-250:]
    es_ci = bootstrap_ci(last_win, historical_es, alpha=0.975)

    line = "=" * 60
    print(line)
    print(f"  MARKET RISK DAILY REPORT  -  {datetime.now():%Y-%m-%d}")
    print(line)

    print("\n  RISK MEASURES (latest day)")
    print(f"    99.0% VaR:          {bt['var99'][-1]:.4%}")
    print(f"    97.5% VaR:          {bt['var975'][-1]:.4%}")
    print(f"    97.5% ES:           {bt['es975'][-1]:.4%}")
    print(f"    97.5% ES 95% CI:    [{es_ci['lo']:.4%}, {es_ci['hi']:.4%}]  (bootstrap)")
    tail_ratio = historical_es(last_win, 0.99) / historical_var(last_win, 0.99)
    print(f"    Tail ratio (99 ES / 99 VaR): {tail_ratio:.2f}x")

    print("\n  STRESSED CALIBRATION")
    label = sw["label"] or "unlabelled high‑volatility window"
    print(f"    Stressed ES:        {sw['stressed_es']:.4%}")
    print(f"    Stress multiplier:  {sw['stress_multiplier']:.2f}x")
    print(f"    Stress period:      {sw['start'].date()} to {sw['end'].date()}")
    print(f"    Identified as:      {label}")

    print(f"\n  BACKTESTING  ({bt['n']} days)")
    print(f"    Exceptions:         {bt['exceptions']} (expected {bt['expected']:.1f})")
    print(f"    Traffic light:      {bt['traffic_light']} (full) | "
          f"{bt['rolling_traffic_light']} (last 250d)")
    print(f"    Kupiec POF:         {'PASS' if bt['kupiec_p']>0.05 else 'FAIL'} (p={bt['kupiec_p']:.3f})")
    print(f"    Cond. coverage:     {'PASS' if bt['cond_cov_p']>0.05 else 'FAIL'} (p={bt['cond_cov_p']:.3f})")
    print(f"    ES Acerbi‑Szekely (97.5%, same‑alpha):")
    print(f"       Z1 = {bt['as_Z1']:+.3f}   Z2 = {bt['as_Z2']:+.3f}  "
          f"({'OK' if bt['as_Z2']<0.10 else 'ES UNDERESTIMATES'})")

    if with_pvalues:
        asp = acerbi_szekely_pvalues(returns, alpha=0.975, n_sims=1000)
        bk = berkowitz_tail(returns, alpha=0.99, n_sims=1000)
        print(f"    ES MC p‑values:    Z1 p={asp['Z1_p']:.3f} | Z2 p={asp['Z2_p']:.3f}")
        print(f"    Berkowitz tail:    mean z={bk['mean_z']:+.3f}, p_mc={bk['p_mc']:.3f}")

    print("\n  MODEL VALIDATION (PLAT, sensitivity‑based)")
    print(f"    Spearman corr:      {plat['spearman']:.3f}")
    print(f"    Unexplained var:    {plat['variance_ratio']:.1%}")
    print(f"    Status:             {plat['zone']}")
    print("\n" + line)
    return bt


def plot_diagnostics(bt, breaches_path="breaches.png", intensity_path="breach_intensity.png"):
    import matplotlib.pyplot as plt

    var, realized, dates = bt["var99"], bt["realized"], bt["dates"]
    exc = bt["exceptions_arr"]

    # 1. Breach scatter plot
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(dates, realized, lw=0.6, color="#888", label="Realized loss")
    ax.plot(dates, var, lw=1.0, color="#185FA5", label="99% VaR")
    ax.scatter(np.asarray(dates)[exc], realized[exc], s=12, color="#E24B4A",
               zorder=5, label="Exceptions")
    ax.set_title("Realized loss vs 99% VaR")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(breaches_path, dpi=130)
    plt.show()
    plt.close(fig)

    # 2. Rolling 60‑day exception count
    win = 60
    intensity = np.convolve(exc.astype(float), np.ones(win), mode="valid")
    idates = np.asarray(dates)[win-1:]
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.fill_between(idates, intensity, color="#E24B4A", alpha=0.35)
    ax.plot(idates, intensity, lw=1.0, color="#B3261E")
    ax.axhline(win * 0.01, ls="--", lw=0.8, color="#444",
               label=f"Expected ({win*0.01:.1f} per {win}d)")
    ax.set_title("Breach intensity (rolling 60‑day exception count)")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(intensity_path, dpi=130)
    plt.show()
    plt.close(fig)

    print(f"saved {breaches_path} and {intensity_path}")

def main():
    print("\n--- Phase 1: data + risk engine ---")
    returns, _ = get_portfolio_returns()
    r = rolling_risk(returns, levels=(0.975, 0.99))
    print(f"Avg 99% VaR {r['var'][0.99].mean():.4%} | Avg 97.5% ES {r['es'][0.975].mean():.4%}")

    print("\n--- Phase 2: backtesting ---")
    bt = run_all(returns)
    print(f"{bt['exceptions']} exceptions / {bt['n']} days | "
          f"traffic light {bt['traffic_light']} | Z2 {bt['as_Z2']:+.3f}")

    print("\n--- Phase 3: stressed window ---")
    sw = find_stressed_window(returns)
    print(f"Worst window {sw['start'].date()}-{sw['end'].date()} "
          f"({sw['label'] or 'unlabelled'}), multiplier {sw['stress_multiplier']:.2f}x")

    print("\n--- Phase 4: full report + charts ---\n")
    bt = generate_report(with_pvalues=True)
    plot_diagnostics(bt)


if __name__ == "__main__":
    main()


Consolidated Analysis: 

The analysis demonstrates, on real market data, that a standard rolling-window historical-simulation model is reactive rather than predictive. It produces an approximately correct long-run breach count and is near capital-neutral under the new ES regime, so by superficial checks it appears sound. Three deeper, independent measurements tell a different and consistent story:

- Breaches are strongly clustered in time (conditional coverage fails on the independence component).
- Tail losses exceed predicted Expected Shortfall (positive Acerbi-Szekely Z2).
- A stressed calibration demands roughly twice the capital of a calm-period calibration (2.12x multiplier).

Taken together, these converging results show that the model under-protects exactly when protection is most needed. This is the precise rationale for FRTB's twin innovations — the move to Expected Shortfall and the requirement for stressed calibration with stricter, timing-aware back testing.

The engine demonstrates a common and dangerous pattern in risk models – they appear well‑calibrated on average, pass naive frequency tests, and earn regulatory green lights from traffic‑light rules, yet they fail more sophisticated timing‑aware and tail‑sensitive tests. The stressed calibration shows that a model built on recent history would have held only 47% of the capital actually required during the COVID‑19 stress. This gap is the fundamental motivation for FRTB’s dual innovations: replacing VaR with Expected Shortfall (to capture tail severity) and mandating a stressed calibration (to avoid window‑dressing with calm data). The conditional coverage failure and positive Acerbi‑Szekely Z2 statistic provide independent, converging evidence of the same problem.



##### **Limitations and next steps:**

The engine uses historical simulation without volatility scaling or regime‑switching – a deliberate baseline. A production IMA model would add liquidity horizons, a full factor model for PLAT, and dynamic volatility weighting (e.g., HS‑Vol‑Weighted). Nonetheless, this self‑contained implementation successfully reproduces the logic and conclusions of the FRTB framework: a model can be “green” on the surface yet still dangerously fragile, and only a combination of stressed calibration, conditional backtests, and ES‑focused diagnostics can reveal the true risk profile.